# Ejercicio 4 — Predicción de abandono y fuga de información del futuro

**Introducción al Machine Learning — Cuarta lista de ejercicios — Daniel Vargas**

Se usa `clientes.csv` para estimar, en una fecha $t_0$, si un cliente
abandonará el servicio en los siguientes 30 días (`abandona_30d`).

In [1]:
import numpy as np
import pandas as pd

from lista4.data import cargar_csv, obtener_xy
from lista4.models import ajustar_regresion_logistica
from lista4.metrics import riesgo_logistico, clasificar, errores_clasificacion

clientes = cargar_csv('clientes.csv')
clientes.head()

,antiguedad_meses,uso_30d_previos,facturas_vencidas_180d,tickets_90d,pago_automatico,ajuste_comercial_30d,abandona_30d
0,66,37.927153,0,1,0,23.039078,0
1,12,46.656223,0,1,1,19.158849,0
2,49,22.990662,0,3,1,27.712278,0
3,49,24.546478,1,0,0,96.997980,1
4,30,64.413222,0,2,1,0.000000,0


## (a) Variables disponibles en $t_0$

Revisando cómo se construye cada variable:

| Variable | Ventana de información | ¿Disponible en $t_0$? |
|---|---|---|
| `antiguedad_meses` | en $t_0$ | Sí |
| `uso_30d_previos` | $t_0-30$ a $t_0$ (pasado) | Sí |
| `facturas_vencidas_180d` | $t_0-180$ a $t_0$ (pasado) | Sí |
| `tickets_90d` | $t_0-90$ a $t_0$ (pasado) | Sí |
| `pago_automatico` | en $t_0$ | Sí |
| `ajuste_comercial_30d` | $t_0$ a $t_0+30$ (**futuro**) | **No** |

Todas usan únicamente información hasta $t_0$, excepto `ajuste_comercial_30d`,
que se construye con datos posteriores a $t_0$ (hasta $t_0+30$): por lo
tanto no puede conocerse en el momento de predecir y no pertenece a
`variables_disponibles`.

In [2]:
variables_disponibles = [
    'antiguedad_meses',
    'uso_30d_previos',
    'facturas_vencidas_180d',
    'tickets_90d',
    'pago_automatico',
]
variables_disponibles

['antiguedad_meses',
 'uso_30d_previos',
 'facturas_vencidas_180d',
 'tickets_90d',
 'pago_automatico']

## (b) Modelos M1 (todas las variables) y M2 (solo `variables_disponibles`)

In [3]:
todas_las_variables = variables_disponibles + ['ajuste_comercial_30d']

X_m1, y = obtener_xy(clientes, todas_las_variables, 'abandona_30d')
X_m2, _ = obtener_xy(clientes, variables_disponibles, 'abandona_30d')

modelo_m1 = ajustar_regresion_logistica(X_m1, y)
modelo_m2 = ajustar_regresion_logistica(X_m2, y)

print('Coeficientes M1:', dict(zip(todas_las_variables, modelo_m1.coef_[0])))
print()
print('Coeficientes M2:', dict(zip(variables_disponibles, modelo_m2.coef_[0])))

Coeficientes M1: {'antiguedad_meses': np.float64(-0.005852670158502702), 'uso_30d_previos': np.float64(-0.007345821827443783), 'facturas_vencidas_180d': np.float64(0.5744464004902786), 'tickets_90d': np.float64(0.09649683772692298), 'pago_automatico': np.float64(-0.5016473658989412), 'ajuste_comercial_30d': np.float64(0.08564359841643515)}

Coeficientes M2: {'antiguedad_meses': np.float64(-0.008387925945716418), 'uso_30d_previos': np.float64(-0.006943928457247933), 'facturas_vencidas_180d': np.float64(0.5418105889109349), 'tickets_90d': np.float64(0.16963628603520248), 'pago_automatico': np.float64(-0.43168690262381537)}


## (c) Riesgo empírico logístico, errores y proporción de errores

In [4]:
p_hat_m1 = modelo_m1.predict_proba(X_m1)[:, 1]
p_hat_m2 = modelo_m2.predict_proba(X_m2)[:, 1]

R_m1 = riesgo_logistico(y, p_hat_m1)
R_m2 = riesgo_logistico(y, p_hat_m2)

clase_m1 = clasificar(p_hat_m1, umbral=0.5)
clase_m2 = clasificar(p_hat_m2, umbral=0.5)

n_err_m1, prop_err_m1 = errores_clasificacion(y, clase_m1)
n_err_m2, prop_err_m2 = errores_clasificacion(y, clase_m2)

print(f'M1 -> riesgo_logistico={R_m1:.4f}  errores={n_err_m1}  proporcion={prop_err_m1:.4f}')
print(f'M2 -> riesgo_logistico={R_m2:.4f}  errores={n_err_m2}  proporcion={prop_err_m2:.4f}')

M1 -> riesgo_logistico=0.3425  errores=160  proporcion=0.1333
M2 -> riesgo_logistico=0.5717  errores=336  proporcion=0.2800


## (d) Tabla resumen

In [5]:
tabla_4 = pd.DataFrame([
    {'modelo': 'M1 (todas las variables)', 'riesgo_logistico': R_m1,
     'n_errores': n_err_m1, 'proporcion_errores': prop_err_m1},
    {'modelo': 'M2 (solo variables_disponibles)', 'riesgo_logistico': R_m2,
     'n_errores': n_err_m2, 'proporcion_errores': prop_err_m2},
])
tabla_4

,modelo,riesgo_logistico,n_errores,proporcion_errores
0,M1 (todas las variables),0.342485,160,0.133333
1,M2 (solo variables_disponibles),0.571716,336,0.280000


## (e) ¿Cuál modelo puede usarse exactamente en $t_0$?

**Respuesta.** Solo **M2** puede ejecutarse en el momento $t_0$: usa
exclusivamente variables conocidas hasta esa fecha. **M1 no puede usarse en
$t_0$** porque depende de `ajuste_comercial_30d`, una variable construida
con información de los 30 días *posteriores* a $t_0$ — en el momento de
predecir, ese dato todavía no existe.

La variable que produce la diferencia entre M1 y M2 es **`ajuste_comercial_30d`**.
Aunque M1 tenga mejor riesgo empírico y menos errores en esta tabla, ese
resultado es engañoso: la mejora proviene de usar información del futuro
(fuga de datos / *data leakage*), algo que no estará disponible al momento
real de hacer la predicción en producción.